# 03 — Cleaning & Feature Engineering

**Project:** F1 Track Personality 2020–2025
**Authors:** Matteo Asscher & Aksel
**Inputs:**
- `data/raw/f1_results_2020_2025.csv` — jolpica API results (2,618 rows × 19 columns, Session 2)
- `data/raw/circuits_wiki_<latest>.parquet` — Wikipedia circuit metadata (30 rows × 12 columns, Session 3)

**Output:** `data/processed/analysis_ready_<timestamp>.parquet` — one row per (season, round, driver), enriched with constructor lineage, position delta, and circuit features. Ready for the heatmap in Session 5.

## Decisions locked in

1. **Constructor lineage = continuous.** We treat ownership-stable rebrands as the same team (Racing Point→Aston Martin, Renault→Alpine, AlphaTauri→RB→Racing Bulls, Alfa Romeo→Sauber→Kick Sauber). Justification: same factory, same chassis lineage, same engineering identity. Without this, several "teams" would only exist for one season, making `season_avg_position` meaningless.
2. **DNF rule.** Drivers with status ≠ Finished/+N Laps get `position = finishers_count + 1`. Punishes DNFs without overweighting them. Alternative (drop DNFs entirely) was rejected because it would bias toward reliable teams.
3. **Aggregation.** Arithmetic mean for `season_avg_position` per (constructor_lineage, season).
4. **Threshold.** Heatmap cells require ≥3 races per (constructor_lineage, circuit) pair. Below threshold = NaN.
5. **Time window.** 2020–2025 (all six seasons). 2025 is partial at time of API pull; noted in data quality.

## Engineered features

| Feature                 | Meaning                                                                                 |
|-------------------------|-----------------------------------------------------------------------------------------|
| `constructor_lineage`   | Continuous team identity across rebrands (see decision 1).                              |
| `effective_position`    | `position` for finishers; `finishers_count + 1` for DNFs.                               |
| `season_avg_position`   | Mean `effective_position` per (constructor_lineage, season), across both drivers.       |
| `position_delta`        | `effective_position − season_avg_position`. Negative = overperformance, positive = under.|
| `track_length_km`       | Float, parsed from Wikipedia `length` string.                                           |
| `turns_count`           | Int, parsed from Wikipedia `turns` string.                                              |
| `circuit_opened_year`   | Int, year extracted from Wikipedia `opened` string.                                     |

In [1]:
from datetime import datetime
from pathlib import Path
import re

import pandas as pd

# Paths
DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print(f"  pandas: {pd.__version__}")
print(f"  raw dir:       {DATA_RAW.resolve()}")
print(f"  processed dir: {DATA_PROCESSED.resolve()}")

Setup OK
  pandas: 3.0.3
  raw dir:       /Users/matteoasscher/AMA Dropbox/Matteo Asscher/Mac/Documents/matteo_aksel/data/raw
  processed dir: /Users/matteoasscher/AMA Dropbox/Matteo Asscher/Mac/Documents/matteo_aksel/data/processed


In [2]:
# Load the API data from Session 2
api_path = DATA_RAW / "f1_results_2020_2025.csv"
api_df = pd.read_csv(api_path)

# Load the latest Wikipedia scrape from Session 3
# We may have multiple timestamped files — pick the most recent one
wiki_candidates = sorted(DATA_RAW.glob("circuits_wiki_*.parquet"))
if not wiki_candidates:
    raise FileNotFoundError("No circuits_wiki_*.parquet file found in data/raw/")
wiki_path = wiki_candidates[-1]
wiki_df = pd.read_parquet(wiki_path)

print(f"API data   : {api_df.shape[0]:>5} rows × {api_df.shape[1]:>2} cols  from  {api_path.name}")
print(f"Wiki data  : {wiki_df.shape[0]:>5} rows × {wiki_df.shape[1]:>2} cols  from  {wiki_path.name}")

print("\nAPI columns:")
print(f"  {list(api_df.columns)}")
print("\nWiki columns:")
print(f"  {list(wiki_df.columns)}")

API data   :  2618 rows × 19 cols  from  f1_results_2020_2025.csv
Wiki data  :    30 rows × 12 cols  from  circuits_wiki_20260524_182752.parquet

API columns:
  ['season', 'round', 'race_name', 'circuit_id', 'circuit_name', 'country', 'date', 'position', 'position_text', 'points', 'grid', 'laps', 'status', 'driver_id', 'driver_code', 'driver_first', 'driver_last', 'constructor_id', 'constructor_name']

Wiki columns:
  ['length', 'turns', 'opened', 'architect', 'capacity', 'location', 'wiki_url', 'circuit_id', 'circuit_name', 'country', 'http_status', 'scrape_error']


In [3]:
# Inspect the messy raw strings we need to clean
print("=" * 70)
print("Wiki — sample of raw strings to clean")
print("=" * 70)

sample = wiki_df[["circuit_id", "length", "turns", "opened"]].head(10)
for _, row in sample.iterrows():
    print(f"\n  {row['circuit_id']}:")
    print(f"    length : {row['length']!r}")
    print(f"    turns  : {row['turns']!r}")
    print(f"    opened : {row['opened']!r}")

print("\n" + "=" * 70)
print("API — `status` value counts (we need this for the DNF rule)")
print("=" * 70)
print(api_df["status"].value_counts().head(20))

print("\n" + "=" * 70)
print("API — `position` and `position_text` for non-finishers (DNF examples)")
print("=" * 70)
# Look at a few rows where status is NOT "Finished" or "+N Lap(s)" 
non_finished = api_df[~api_df["status"].str.contains("Finished|Lap", na=False, regex=True)]
print(non_finished[["status", "position", "position_text"]].head(10).to_string(index=False))

Wiki — sample of raw strings to clean

  red_bull_ring:
    length : '5.911\xa0km (3.673\xa0mi)'
    turns  : '16'
    opened : '26\xa0July 1969 ; 56 years ago ( 1969-07-26 ) Re-opened: 15\xa0May 2011 ; 15 years ago ( 2011-05-15 )'

  hungaroring:
    length : '4.014\xa0km (2.494\xa0mi)'
    turns  : '16'
    opened : '24\xa0March 1986 ; 40 years ago ( 1986-03-24 )'

  silverstone:
    length : '5.907\xa0km (3.670\xa0mi)'
    turns  : '17'
    opened : '1948'

  catalunya:
    length : '4.747\xa0km (2.950\xa0mi)'
    turns  : '14'
    opened : '10\xa0September 1991 ; 34 years ago ( 1991-09-10 )'

  spa:
    length : '14.982\xa0km (9.309\xa0mi)'
    turns  : '25'
    opened : 'August\xa01921 ; 104\xa0years ago ( 1921-08 )'

  monza:
    length : '10.000\xa0km (6.214\xa0mi)'
    turns  : '9'
    opened : '3\xa0September 1922 ; 103 years ago ( 1922-09-03 )'

  mugello:
    length : '64.591\xa0km (40.135\xa0mi)'
    turns  : '400+'
    opened : '23\xa0June 1974 ; 51 years ago ( 1974-06-23 

In [4]:
# Multi-layout circuits patch — Wikipedia infobox grabbed historic/non-F1 layouts.
# Verified against F1 race result pages (2020 Tuscan GP, 2020 Eifel GP, current
# Monza GP layout, modern Spa GP Circuit). All four are documented in data quality.

LAYOUT_PATCHES = {
    "monza":       {"length": "5.793 km (3.600 mi)", "turns": "11"},
    "mugello":     {"length": "5.245 km (3.259 mi)", "turns": "15"},
    "nurburgring": {"length": "5.148 km (3.199 mi)", "turns": "15"},
    "spa":         {"length": "7.004 km (4.352 mi)", "turns": "19"},
}

print("Before patch:")
print(wiki_df[wiki_df["circuit_id"].isin(LAYOUT_PATCHES)][["circuit_id", "length", "turns"]].to_string(index=False))

for cid, fixes in LAYOUT_PATCHES.items():
    mask = wiki_df["circuit_id"] == cid
    for col, val in fixes.items():
        wiki_df.loc[mask, col] = val

print("\nAfter patch:")
print(wiki_df[wiki_df["circuit_id"].isin(LAYOUT_PATCHES)][["circuit_id", "length", "turns"]].to_string(index=False))

Before patch:
 circuit_id                length turns
        spa  14.982 km (9.309 mi)    25
      monza  10.000 km (6.214 mi)     9
    mugello 64.591 km (40.135 mi)  400+
nurburgring 28.265 km (17.563 mi)   187

After patch:
 circuit_id              length turns
        spa 7.004 km (4.352 mi)    19
      monza 5.793 km (3.600 mi)    11
    mugello 5.245 km (3.259 mi)    15
nurburgring 5.148 km (3.199 mi)    15


In [5]:
# Parse the messy raw strings into clean numeric columns

def parse_track_length_km(raw: str) -> float | None:
    """
    Extract the km value from strings like '5.412 km (3.363 mi)'.
    Returns None if the input is NaN or the pattern doesn't match.
    """
    if pd.isna(raw):
        return None
    # Match a number followed by optional whitespace and 'km'.
    # \xa0 = non-breaking space (Wikipedia uses it instead of regular space).
    match = re.search(r"([\d.]+)\s*km", raw.replace("\xa0", " "))
    return float(match.group(1)) if match else None


def parse_turns_count(raw: str) -> int | None:
    """
    Extract the first integer from strings like '15' or '15 (5 layouts)'.
    Returns None if no integer found.
    """
    if pd.isna(raw):
        return None
    match = re.search(r"\d+", str(raw))
    return int(match.group(0)) if match else None


def parse_opened_year(raw: str) -> int | None:
    """
    Extract a 4-digit year from messy date strings like:
      '17 March 2004 ; 21 years ago ( 2004-03-17 )'
      '1948'
      'August 1921 ; 104 years ago ( 1921-08 )'
    Returns None if no 4-digit year found.
    """
    if pd.isna(raw):
        return None
    # Find the FIRST 4-digit number between 1900 and 2030 (sanity bound).
    match = re.search(r"\b(19\d{2}|20[0-2]\d)\b", str(raw))
    return int(match.group(0)) if match else None


# Apply each parser to create new clean columns
wiki_df["track_length_km"]    = wiki_df["length"].apply(parse_track_length_km)
wiki_df["turns_count"]        = wiki_df["turns"].apply(parse_turns_count)
wiki_df["circuit_opened_year"] = wiki_df["opened"].apply(parse_opened_year)

# Quick QA: did anything fail to parse?
print("Parsing QA:")
print(f"  track_length_km    : {wiki_df['track_length_km'].notna().sum():>2}/30 parsed, {wiki_df['track_length_km'].isna().sum()} null")
print(f"  turns_count        : {wiki_df['turns_count'].notna().sum():>2}/30 parsed, {wiki_df['turns_count'].isna().sum()} null")
print(f"  circuit_opened_year: {wiki_df['circuit_opened_year'].notna().sum():>2}/30 parsed, {wiki_df['circuit_opened_year'].isna().sum()} null")

# Show the result for all 30 circuits
print("\nCleaned circuits:")
print(wiki_df[["circuit_id", "track_length_km", "turns_count", "circuit_opened_year"]].to_string(index=False))

Parsing QA:
  track_length_km    : 30/30 parsed, 0 null
  turns_count        : 30/30 parsed, 0 null
  circuit_opened_year: 29/30 parsed, 1 null

Cleaned circuits:
   circuit_id  track_length_km  turns_count  circuit_opened_year
red_bull_ring            5.911           16               1969.0
  hungaroring            4.014           16               1986.0
  silverstone            5.907           17               1948.0
    catalunya            4.747           14               1991.0
          spa            7.004           19               1921.0
        monza            5.793           11               1922.0
      mugello            5.245           15               1974.0
        sochi            5.848           18               2014.0
  nurburgring            5.148           15                  NaN
     portimao            4.684           16               2008.0
        imola            5.018           12               1953.0
     istanbul            1.346           14              

In [6]:
# Investigate why some lengths look wrong (e.g. istanbul 1.346, yas_marina 2.360)
# Show the raw `length` field for the suspicious ones AND a few that look correct.

suspicious = ["istanbul", "yas_marina", "miami", "jeddah", "shanghai"]
correct    = ["bahrain", "silverstone", "monza"]

print("SUSPICIOUS (parsed length looks too small):")
for cid in suspicious:
    raw = wiki_df.loc[wiki_df["circuit_id"] == cid, "length"].iloc[0]
    parsed = wiki_df.loc[wiki_df["circuit_id"] == cid, "track_length_km"].iloc[0]
    print(f"  {cid:<12} parsed={parsed}  raw={raw!r}")

print("\nCONTROL (these look right):")
for cid in correct:
    raw = wiki_df.loc[wiki_df["circuit_id"] == cid, "length"].iloc[0]
    parsed = wiki_df.loc[wiki_df["circuit_id"] == cid, "track_length_km"].iloc[0]
    print(f"  {cid:<12} parsed={parsed}  raw={raw!r}")

SUSPICIOUS (parsed length looks too small):
  istanbul     parsed=1.346  raw='1.346\xa0km (0.836\xa0mi)'
  yas_marina   parsed=2.36  raw='2.360\xa0km (1.466\xa0mi)'
  miami        parsed=2.32  raw='1.442\xa0mi (2.320\xa0km)'
  jeddah       parsed=3.45  raw='3.450\xa0km (2.144\xa0mi)'
  shanghai     parsed=3.051  raw='3.051\xa0km (1.896\xa0mi)'

CONTROL (these look right):
  bahrain      parsed=5.417  raw='5.417\xa0km (3.366\xa0mi)'
  silverstone  parsed=5.907  raw='5.907\xa0km (3.670\xa0mi)'
  monza        parsed=5.793  raw='5.793 km (3.600 mi)'


In [7]:
    # Comprehensive layout patch — verified against F1 race result Wikipedia pages
# (e.g. "2024 Bahrain GP", "2024 Italian GP") which list the actual GP layout used.
#
# Root cause: Wikipedia circuit pages list multiple historical layouts; our 
# scraper grabbed the first one which is often not the current F1 layout.
# This patch replaces ALL length/turns values with verified GP layout values.
# 16 of 30 circuits needed correction. Documented in data quality notes.

VERIFIED_LAYOUTS = {
    # circuit_id      length_km  turns  (source: Wikipedia race page for 2024 GP)
    "red_bull_ring":    (4.318, 10),
    "hungaroring":      (4.381, 14),
    "silverstone":      (5.891, 18),
    "catalunya":        (4.657, 14),  # post-2023 layout w/o chicane
    "spa":              (7.004, 19),
    "monza":            (5.793, 11),
    "mugello":          (5.245, 15),
    "sochi":            (5.848, 18),
    "nurburgring":      (5.148, 15),
    "portimao":         (4.653, 15),
    "imola":            (4.909, 19),
    "istanbul":         (5.338, 14),
    "bahrain":          (5.412, 15),
    "yas_marina":       (5.281, 16),  # post-2021 redesign (5/6 seasons)
    "monaco":           (3.337, 19),
    "baku":             (6.003, 20),
    "ricard":           (5.842, 15),
    "zandvoort":        (4.259, 14),
    "americas":         (5.513, 20),
    "rodriguez":        (4.304, 17),
    "interlagos":       (4.309, 15),
    "losail":           (5.419, 16),
    "jeddah":           (6.174, 27),
    "albert_park":      (5.278, 14),
    "miami":            (5.412, 19),
    "villeneuve":       (4.361, 14),
    "marina_bay":       (4.940, 19),  # post-2023 layout
    "suzuka":           (5.807, 18),
    "vegas":            (6.201, 17),
    "shanghai":         (5.451, 16),
}

# Apply: overwrite track_length_km and turns_count for all circuits
print(f"Patching {len(VERIFIED_LAYOUTS)} circuits with verified F1 GP layout values...\n")

for cid, (length_km, turns) in VERIFIED_LAYOUTS.items():
    mask = wiki_df["circuit_id"] == cid
    if not mask.any():
        print(f"  ⚠️  {cid} not found in wiki_df!")
        continue
    wiki_df.loc[mask, "track_length_km"] = length_km
    wiki_df.loc[mask, "turns_count"]     = turns

print("Final cleaned circuits:")
print(wiki_df[["circuit_id", "track_length_km", "turns_count", "circuit_opened_year"]].to_string(index=False))

Patching 30 circuits with verified F1 GP layout values...

Final cleaned circuits:
   circuit_id  track_length_km  turns_count  circuit_opened_year
red_bull_ring            4.318           10               1969.0
  hungaroring            4.381           14               1986.0
  silverstone            5.891           18               1948.0
    catalunya            4.657           14               1991.0
          spa            7.004           19               1921.0
        monza            5.793           11               1922.0
      mugello            5.245           15               1974.0
        sochi            5.848           18               2014.0
  nurburgring            5.148           15                  NaN
     portimao            4.653           15               2008.0
        imola            4.909           19               1953.0
     istanbul            5.338           14               2005.0
      bahrain            5.412           15               2004.0
   yas_

In [8]:
# Constructor lineage mapping — continuous team identity across rebrands.
# See decision 1 in the header markdown for justification.
#
# Key:   constructor_id from jolpica API  (e.g. 'racing_point', 'aston_martin')
# Value: lineage name we use for grouping (e.g. 'Aston Martin')

LINEAGE = {
    # Stable teams (kept the same id and name 2020-2025)
    "mercedes":        "Mercedes",
    "red_bull":        "Red Bull",
    "ferrari":         "Ferrari",
    "mclaren":         "McLaren",
    "williams":        "Williams",
    "haas":            "Haas",
    
    # Racing Point (2020) -> Aston Martin (2021-)
    "racing_point":    "Aston Martin",
    "aston_martin":    "Aston Martin",
    
    # Renault (2020) -> Alpine (2021-)
   
    "renault":         "Alpine",
    "alpine":          "Alpine",
    
    # AlphaTauri (2020-2023) -> RB (2024) -> Racing Bulls (2025)
    "alphatauri":      "AlphaTauri/RB",
    "rb":              "AlphaTauri/RB",
    "racing_bulls":    "AlphaTauri/RB",
    
    # Alfa Romeo (2020-2023) -> Sauber (2024) -> Kick Sauber (2025)
    "alfa":            "Sauber",
    "sauber":          "Sauber",
    "kick_sauber":     "Sauber",
}

# Show all unique constructor_ids in the API data so we can check we covered everyone
unique_constructors = sorted(api_df["constructor_id"].unique())
print(f"Unique constructor_ids in API: {len(unique_constructors)}\n")
print("Constructor coverage:")
for cid in unique_constructors:
    lineage = LINEAGE.get(cid, "❌ MISSING")
    print(f"  {cid:<18} → {lineage}")

Unique constructor_ids in API: 14

Constructor coverage:
  alfa               → Sauber
  alphatauri         → AlphaTauri/RB
  alpine             → Alpine
  aston_martin       → Aston Martin
  ferrari            → Ferrari
  haas               → Haas
  mclaren            → McLaren
  mercedes           → Mercedes
  racing_point       → Aston Martin
  rb                 → AlphaTauri/RB
  red_bull           → Red Bull
  renault            → Alpine
  sauber             → Sauber
  williams           → Williams


In [9]:
# Apply the lineage mapping to create the constructor_lineage column

api_df["constructor_lineage"] = api_df["constructor_id"].map(LINEAGE)

# Sanity check: no NaN values
n_missing = api_df["constructor_lineage"].isna().sum()
print(f"Rows with missing constructor_lineage: {n_missing} (expect 0)\n")

# Count rows per lineage
print("Race entries per constructor lineage:")
print(api_df["constructor_lineage"].value_counts().to_string())

# Verify the merges happened correctly: which raw IDs map to each lineage?
print("\nLineage composition (raw constructor_id → constructor_lineage):")
composition = api_df.groupby("constructor_lineage")["constructor_id"].unique().sort_index()
for lineage, ids in composition.items():
    print(f"  {lineage:<18}: {sorted(ids)}")

Rows with missing constructor_lineage: 0 (expect 0)

Race entries per constructor lineage:
constructor_lineage
Mercedes         262
Ferrari          262
McLaren          262
AlphaTauri/RB    262
Alpine           262
Sauber           262
Red Bull         262
Haas             262
Aston Martin     261
Williams         261

Lineage composition (raw constructor_id → constructor_lineage):
  AlphaTauri/RB     : ['alphatauri', 'rb']
  Alpine            : ['alpine', 'renault']
  Aston Martin      : ['aston_martin', 'racing_point']
  Ferrari           : ['ferrari']
  Haas              : ['haas']
  McLaren           : ['mclaren']
  Mercedes          : ['mercedes']
  Red Bull          : ['red_bull']
  Sauber            : ['alfa', 'sauber']
  Williams          : ['williams']


In [10]:
# DNF rule: compute effective_position per race.
# Drivers with status indicating they finished (Finished, Lapped, +N Lap(s)) 
# keep their position. Everyone else (Engine failure, Collision, etc.) gets
# finishers_count + 1 — punishes DNFs without overweighting them.

# Step 1: classify each row as finisher or not
FINISHED_STATUSES = {"Finished", "Lapped"}
api_df["is_finisher"] = (
    api_df["status"].isin(FINISHED_STATUSES) 
    | api_df["status"].str.match(r"^\+\d+ Lap", na=False)
)

# Step 2: for each race, count finishers and compute DNF position
race_groups = api_df.groupby(["season", "round"], as_index=False)
finisher_counts = race_groups["is_finisher"].sum().rename(columns={"is_finisher": "finishers_count"})

# Merge the finisher count back into the main df
api_df = api_df.merge(finisher_counts, on=["season", "round"], how="left")

# Step 3: compute effective_position
# - For finishers: use the original numeric position
# - For non-finishers: use finishers_count + 1
api_df["effective_position"] = api_df["position"].where(
    api_df["is_finisher"], 
    api_df["finishers_count"] + 1
)

# QA: sanity check the rule
print(f"Total race entries: {len(api_df):,}")
print(f"Finishers          : {api_df['is_finisher'].sum():,}")
print(f"Non-finishers (DNF): {(~api_df['is_finisher']).sum():,}")

print("\nDNF rule applied — sample of 5 DNF rows:")
dnfs_sample = api_df[~api_df["is_finisher"]][
    ["season", "round", "driver_id", "constructor_lineage", "status", "position", "finishers_count", "effective_position"]
].head(8)
print(dnfs_sample.to_string(index=False))

print("\nVerification — same race (2020 Austrian GP, round 1):")
race_check = api_df[(api_df["season"] == 2020) & (api_df["round"] == 1)][
    ["driver_id", "status", "position", "is_finisher", "finishers_count", "effective_position"]
].sort_values("position")
print(race_check.to_string(index=False))

Total race entries: 2,618
Finishers          : 2,248
Non-finishers (DNF): 370

DNF rule applied — sample of 5 DNF rows:
 season  round       driver_id constructor_lineage        status  position  finishers_count  effective_position
   2020      1           kvyat       AlphaTauri/RB    Suspension        12               11                  12
   2020      1           albon            Red Bull   Electronics        13               11                  12
   2020      1       raikkonen              Sauber         Wheel        14               11                  12
   2020      1         russell            Williams Fuel pressure        15               11                  12
   2020      1        grosjean                Haas        Brakes        16               11                  12
   2020      1 kevin_magnussen                Haas        Brakes        17               11                  12
   2020      1          stroll        Aston Martin        Engine        18               11     

In [11]:
# Compute season_avg_position per (constructor_lineage, season) — Decision 3
# Then position_delta = effective_position - season_avg_position

# Step 1: season average per lineage
season_avg = (
    api_df.groupby(["constructor_lineage", "season"], as_index=False)["effective_position"]
    .mean()
    .rename(columns={"effective_position": "season_avg_position"})
)

print(f"Season averages computed: {len(season_avg)} (constructor_lineage, season) pairs\n")

# Show all season averages — quick sanity check
print("Season averages (round-by-round mean of effective_position per lineage):")
pivot = season_avg.pivot(index="constructor_lineage", columns="season", values="season_avg_position").round(2)
print(pivot.to_string())

# Step 2: merge season_avg back into the main df
api_df = api_df.merge(season_avg, on=["constructor_lineage", "season"], how="left")

# Step 3: compute position_delta
api_df["position_delta"] = api_df["effective_position"] - api_df["season_avg_position"]

# Sanity check on delta: per lineage-season, the deltas should sum to ~0 (it's a mean-deviation)
print("\nSanity check: per-lineage-season, mean of position_delta should be ~0")
delta_check = api_df.groupby(["constructor_lineage", "season"])["position_delta"].mean().round(4)
print(f"  Min mean delta : {delta_check.min():.6f}")
print(f"  Max mean delta : {delta_check.max():.6f}")
print(f"  (should be 0 by construction — any non-zero is floating point noise)")

# Show a few examples
print("\nExample rows — Red Bull 2023:")
example = api_df[(api_df["constructor_lineage"] == "Red Bull") & (api_df["season"] == 2023)][
    ["round", "driver_id", "effective_position", "season_avg_position", "position_delta"]
].head(10)
print(example.round(2).to_string(index=False))

Season averages computed: 60 (constructor_lineage, season) pairs

Season averages (round-by-round mean of effective_position per lineage):
season                2020   2021   2022   2023   2024   2025
constructor_lineage                                          
AlphaTauri/RB        10.38  11.00  13.18  13.45  13.15  12.02
Alpine                8.56  10.09   9.64  11.05  13.17  15.02
Aston Martin          8.91  11.77  11.75   8.66  11.62  12.66
Ferrari              10.44   7.00   6.34   7.45   5.06   7.48
Haas                 15.21  17.00  13.98  14.80  12.21  12.04
McLaren               8.41   7.77  10.45   9.32   4.71   4.08
Mercedes              3.65   5.30   5.43   6.93   6.81   7.15
Red Bull              7.18   5.66   4.34   3.41   6.44   8.08
Sauber               13.59  13.07  12.77  13.82  15.60  13.23
Williams             15.38  14.45  14.55  13.73  14.77  11.42

Sanity check: per-lineage-season, mean of position_delta should be ~0
  Min mean delta : -0.000000
  Max mean delta 

In [12]:
# Join the cleaned API data with the cleaned Wikipedia metadata.
# Key: circuit_id (present in both, exact match)

# Pick only the clean columns from wiki_df — drop the raw scraped strings and diagnostics
wiki_for_join = wiki_df[[
    "circuit_id", 
    "track_length_km", 
    "turns_count", 
    "circuit_opened_year",
]]

# Sanity: both sides should have circuit_id with no nulls
assert api_df["circuit_id"].notna().all(), "Some API rows missing circuit_id"
assert wiki_for_join["circuit_id"].notna().all(), "Some Wiki rows missing circuit_id"

# Check all API circuit_ids are in wiki (otherwise we'd lose rows on inner join)
missing_in_wiki = set(api_df["circuit_id"]) - set(wiki_for_join["circuit_id"])
print(f"API circuits not in Wiki: {len(missing_in_wiki)}  (expect 0)")
if missing_in_wiki:
    print(f"  Missing: {missing_in_wiki}")

# Left join — keep every API row, attach Wiki fields where available
analysis_df = api_df.merge(wiki_for_join, on="circuit_id", how="left")

print(f"\nBefore join: api_df = {api_df.shape}, wiki_for_join = {wiki_for_join.shape}")
print(f"After join : analysis_df = {analysis_df.shape}")
print(f"Rows lost or duplicated: {len(analysis_df) - len(api_df)}  (expect 0)")

# Final QA: any rows with missing track_length_km?
n_missing_length = analysis_df["track_length_km"].isna().sum()
n_missing_turns  = analysis_df["turns_count"].isna().sum()
print(f"\nRows missing track_length_km: {n_missing_length}")
print(f"Rows missing turns_count    : {n_missing_turns}")

# Show the final column set
print(f"\nFinal columns ({len(analysis_df.columns)}):")
for col in analysis_df.columns:
    dtype = analysis_df[col].dtype
    print(f"  {col:<25} ({dtype})")

API circuits not in Wiki: 0  (expect 0)

Before join: api_df = (2618, 25), wiki_for_join = (30, 4)
After join : analysis_df = (2618, 28)
Rows lost or duplicated: 0  (expect 0)

Rows missing track_length_km: 0
Rows missing turns_count    : 0

Final columns (28):
  season                    (int64)
  round                     (int64)
  race_name                 (str)
  circuit_id                (str)
  circuit_name              (str)
  country                   (str)
  date                      (str)
  position                  (int64)
  position_text             (str)
  points                    (float64)
  grid                      (int64)
  laps                      (int64)
  status                    (str)
  driver_id                 (str)
  driver_code               (str)
  driver_first              (str)
  driver_last               (str)
  constructor_id            (str)
  constructor_name          (str)
  constructor_lineage       (str)
  is_finisher               (bool)
  finishe

In [13]:
# Save the analysis-ready table to data/processed/ with a timestamp
# (same pattern as Session 3 — never overwrite, always keep history)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

out_path = DATA_PROCESSED / f"analysis_ready_{ts}.parquet"
analysis_df.to_parquet(out_path, index=False)

# Also save a CSV for human readability (smaller analysis tables don't need both,
# but the brief asks for both raw formats so we maintain the pattern)
csv_path = DATA_PROCESSED / f"analysis_ready_{ts}.csv"
analysis_df.to_csv(csv_path, index=False)

print(f"Saved analysis-ready table:")
print(f"  {out_path.name}  ({out_path.stat().st_size:>8,} bytes)")
print(f"  {csv_path.name}  ({csv_path.stat().st_size:>8,} bytes)")
print(f"\nShape: {analysis_df.shape[0]:,} rows × {analysis_df.shape[1]} columns")
print(f"Memory: {analysis_df.memory_usage(deep=True).sum() / 1024:.1f} KB in-memory")

Saved analysis-ready table:
  analysis_ready_20260528_155210.parquet  (  51,207 bytes)
  analysis_ready_20260528_155210.csv  ( 570,506 bytes)

Shape: 2,618 rows × 28 columns
Memory: 870.3 KB in-memory


## Data Dictionary

Analysis-ready table saved to `data/processed/analysis_ready_<timestamp>.parquet` and `.csv`. **2,618 rows × 28 columns**, one row per (season, round, driver).

### Identity & race context

| Column | Type | Source | Description |
|---|---|---|---|
| `season` | int | jolpica API | Year, 2020–2025. |
| `round` | int | jolpica API | Round number within the season. |
| `race_name` | str | jolpica API | Full GP name, e.g. "Bahrain Grand Prix". |
| `circuit_id` | str | jolpica API | Ergast-style key, e.g. `bahrain`. Join key with Wikipedia data. |
| `circuit_name` | str | jolpica API | Full circuit name, e.g. "Bahrain International Circuit". |
| `country` | str | jolpica API | Country, e.g. "Bahrain". |
| `date` | str | jolpica API | Race date as YYYY-MM-DD string. |

### Driver & constructor

| Column | Type | Source | Description |
|---|---|---|---|
| `driver_id` | str | jolpica API | Ergast-style key, e.g. `max_verstappen`. |
| `driver_code` | str | jolpica API | Three-letter code, e.g. `VER`. |
| `driver_first`, `driver_last` | str | jolpica API | First and last name. |
| `constructor_id` | str | jolpica API | Raw constructor id, e.g. `racing_point`, `aston_martin`. |
| `constructor_name` | str | jolpica API | Display name as used that season. |
| **`constructor_lineage`** | str | engineered (Cell 8–9) | Continuous team identity across rebrands. Decision 1. |

### Race outcome

| Column | Type | Source | Description |
|---|---|---|---|
| `position` | int | jolpica API | Raw finishing position from the API. |
| `position_text` | str | jolpica API | Raw position label, sometimes `"R"` for retired. |
| `points` | float | jolpica API | Championship points awarded. |
| `grid` | int | jolpica API | Starting grid position. |
| `laps` | int | jolpica API | Laps completed. |
| `status` | str | jolpica API | Result status: `Finished`, `+1 Lap`, `Engine`, `Collision`, etc. |
| `is_finisher` | bool | engineered (Cell 10) | `True` if `status` is `Finished`, `Lapped`, or `+N Lap(s)`. |
| `finishers_count` | int | engineered (Cell 10) | Number of classified finishers in this race. |
| **`effective_position`** | int | engineered (Cell 10) | DNF rule applied: finishers keep `position`; non-finishers get `finishers_count + 1`. Decision 2. |

### Performance metrics (used by the heatmap)

| Column | Type | Source | Description |
|---|---|---|---|
| **`season_avg_position`** | float | engineered (Cell 11) | Mean `effective_position` per (constructor_lineage, season). Decision 3. |
| **`position_delta`** | float | engineered (Cell 11) | `effective_position − season_avg_position`. **Negative = overperformance, positive = underperformance.** Core feature for the heatmap. |

### Circuit features (from Wikipedia)

| Column | Type | Source | Description |
|---|---|---|---|
| `track_length_km` | float | Wikipedia + manual verification | Length of the current F1 layout in kilometers. |
| `turns_count` | int | Wikipedia + manual verification | Number of corners on the current F1 layout. |
| `circuit_opened_year` | float | Wikipedia (parsed) | 4-digit year the circuit opened. `NaN` for Nürburgring (missing on source page). |

## Data Quality Notes

Issues we discovered and how we handled them. Each is documented in the notebook with the relevant cell number for traceability.

1. **Wikipedia "Length" and "Turns" grabbed wrong layouts for many circuits.** Wikipedia circuit infoboxes list multiple historical layouts; our scraper picked the first one regardless of whether F1 currently uses it. Affected ~16 of 30 circuits, including Monza, Mugello, Nürburgring, Spa, Miami, Istanbul, Yas Marina, Jeddah, and Shanghai. **Fix (Cell 7):** built a `VERIFIED_LAYOUTS` table with the canonical 2024 F1 GP layout values cross-referenced against Wikipedia race result pages (e.g. `2024_Bahrain_Grand_Prix`). All 30 circuits now have authoritative values.

2. **Yas Marina layout changed in 2021.** The old 5.554 km / 21-turn layout was used for the 2020 Abu Dhabi GP only; 2021–2025 used the new 5.281 km / 16-turn layout. We use the modern layout (matches 5 of 6 seasons). Equivalent layout-change ambiguity exists for Bahrain (Sakhir GP outer layout in 2020), Marina Bay (post-2023 changes), and Catalunya (post-2023 changes). In each case we use the dominant layout.

3. **Nürburgring `circuit_opened_year` is NaN.** Our parser found no 4-digit year on the page because the infobox doesn't have an "Opened" row matching our regex (the page has separate "Built" sections for Nordschleife and GP-Strecke). Acceptable gap: we only have 1 race at this circuit in 2020–2025, and the feature isn't on the critical path.

4. **Vegas required a URL override.** The jolpica API returned the *Grand Prix event* page URL (`Las_Vegas_Grand_Prix#Circuit`) instead of the *circuit* page (`Las_Vegas_Strip_Circuit`). Patched in Session 3.

5. **jolpica still uses old constructor IDs for 2025 rebrands.** AlphaTauri's 2025 rebrand to Racing Bulls is reported as `rb`; Sauber's 2025 rebrand to Kick Sauber is reported as `sauber`. Our `LINEAGE` mapping handles both naming conventions transparently.

6. **2025 season is partial at API pull time.** The API returns whatever races have been completed; 2025 finishes after our scrape. Means 2025 has fewer rows per lineage than other seasons and `season_avg_position` for 2025 is computed on a partial sample.

7. **Aston Martin and Williams each have 261 rows instead of the expected 262.** One driver-race entry is missing somewhere — likely a driver substitution or DNS recorded without a results entry. Minor (0.4% of those teams' data), no impact on aggregates.

8. **30 circuits with ≥3 race threshold means many cells in the heatmap will be NaN.** Decision 4 requires ≥3 races per (constructor_lineage, circuit) pair. Many circuits only hosted 1–2 GPs in 2020–2025 (Mugello, Portimão, Istanbul, Sochi, etc.). The heatmap will have well-populated columns for circuits that ran every season (Bahrain, Silverstone, Monaco, etc.) and sparse columns elsewhere. This is a property of the data, not a defect.